# EEG 신호 & EEGNet 디코더 input/output 탐색

`neural-to-output` 프로젝트와 독립적인 sandbox입니다 (`pyproject.toml`/`uv.lock` 미수정, `sandbox/`는 `.gitignore`된 폴더).

각 단계에서 실제 EEG 데이터가 어떤 shape으로 흐르는지, `EEGNet` 디코더의 입력/출력이 어떻게 되는지 직접 출력해서 확인합니다.

사용 데이터셋: BCI Competition IV 2a (`BNCI2014_001`, moabb로 로드), subject 1만.

In [ ]:
import warnings

import matplotlib.animation as animation
import matplotlib.pyplot as plt
import mne
import numpy as np
import torch
from IPython.display import HTML, Markdown, display

from braindecode.datasets import MOABBDataset
from braindecode.models import EEGNet
from braindecode.preprocessing import (
    Preprocessor,
    create_windows_from_events,
    exponential_moving_standardize,
    preprocess,
)

warnings.filterwarnings("ignore")  # mne/braindecode의 사소한 UserWarning 숨김
mne.set_log_level("ERROR")  # mne의 시끄러운 필터 설계 로그(INFO) 숨김

## 1. 원본 EEG 신호 로드

`n2o.signal.dataset.EEG.read()`가 실제로 반환해야 할 것과 같은 종류의 데이터: 채널(ch) 수, 샘플링 레이트(sfreq), raw 신호의 shape을 확인합니다.

In [ ]:
dataset = MOABBDataset(dataset_name="BNCI2014_001", subject_ids=[1])

raw = dataset.datasets[0].raw
print("n_channels:", len(raw.ch_names))
print("ch_names:", raw.ch_names)
print("sfreq (Hz):", raw.info["sfreq"])
print("raw data shape (n_channels, n_times):", raw.get_data().shape)

In [ ]:
display(Markdown("### 원본 신호를 눈으로 보기 — 숫자(shape, 값)만 보면 감이 안 오니, 실제 파형과 전극 위치를 그려봅니다."))

picks = ["Fz", "C3", "Cz", "C4", "Pz"]
seconds = 5
sfreq_before = raw.info["sfreq"]
n_samples = int(seconds * sfreq_before)
data = raw.get_data(picks=picks)[:, :n_samples]
times = np.arange(n_samples) / sfreq_before

fig, axes = plt.subplots(len(picks), 1, figsize=(10, 6), sharex=True)
for ax, ch_name, ch_data in zip(axes, picks, data):
    ax.plot(times, ch_data, linewidth=0.8)
    ax.set_ylabel(ch_name, rotation=0, labelpad=25)
    ax.set_yticks([])
axes[-1].set_xlabel("time (s)")
fig.suptitle(f"Raw EEG waveform (first {seconds}s, 5 channels)")
plt.tight_layout()
plt.show()

# 다음 단계(전처리) 전/후 비교에 쓸 원본 값을 따로 저장
raw_c3_before = raw.get_data(picks=["C3"])[0, :n_samples].copy()

fig = mne.viz.plot_sensors(raw.info, show_names=True)
fig.suptitle("Electrode positions (top-down view)")
plt.show()

## 2. 전처리 (밴드패스 필터 + 표준화)

EEG 원본 신호에는 눈 움직임/근육 잡음 같은 저주파·고주파 잡음이 섞여 있어서, 그대로 모델에 넣으면 학습이 잘 안 됩니다. 그래서 보통 두 가지를 합니다:

1. **밴드패스 필터 (4-38Hz)**: 운동 상상(motor imagery)과 관련된 뇌파 성분이 주로 이 대역에 있어서, 그 바깥의 저주파 표류·고주파 잡음을 걸러냅니다.
2. **표준화 (exponential moving standardization)**: 채널/시간마다 신호 크기가 들쭉날쭉하니, 평균 0·분산 1 근처로 스케일을 맞춰서 모델이 크기 차이에 휘둘리지 않게 합니다.

아래에서 전/후를 직접 비교해봅니다.

In [ ]:
def to_microvolts(data):
    return data * 1e6


preprocessors = [
    Preprocessor("pick_types", apply_on_array=False, eeg=True, meg=False, stim=False),
    Preprocessor(to_microvolts),  # V -> uV
    Preprocessor("filter", apply_on_array=False, l_freq=4.0, h_freq=38.0),
    Preprocessor(exponential_moving_standardize, factor_new=1e-3, init_block_size=1000),
]
preprocess(dataset, preprocessors)

raw = dataset.datasets[0].raw
print("preprocessed n_channels (EOG 채널 제외 후):", len(raw.ch_names))
print("preprocessed data shape:", raw.get_data().shape)

In [ ]:
display(Markdown(
    "### 전처리 전/후 비교 — 뭐가 바뀌었는지 구체적으로 보기\n\n"
    "**밴드패스 필터(4-38Hz)**: 저주파(느린 표류, 눈 움직임 등)와 고주파(근육 잡음 등) 성분을 제거하고 4-38Hz 대역만 남깁니다. "
    "**표준화**는 그 결과를 평균 0, 표준편차 1 근처로 스케일만 맞춥니다.\n\n"
    "그래서 위/아래 그래프의 **y축 단위 자체가 다릅니다** — 위는 실제 물리 단위(µV), 아래는 '평균 대비 몇 표준편차'인 상대값입니다. "
    "시간 파형만으로는 차이가 미묘해 보일 수 있어서, 숫자 요약과 주파수 성분 비교 그래프를 같이 봅니다 — "
    "특히 두 번째 그래프에서 초록색 영역(4-38Hz) 밖의 파워가 필터링 후 확 줄어드는 걸 확인하세요."
))

raw_c3_before_uv = raw_c3_before * 1e6  # V -> uV, "전" 쪽도 같은 단위로 맞춰서 비교
raw_c3_after = raw.get_data(picks=["C3"])[0, :len(raw_c3_before)]
t = np.arange(len(raw_c3_before)) / sfreq_before

print(f"전처리 전 (unit: uV):        mean={raw_c3_before_uv.mean():+8.2f}  std={raw_c3_before_uv.std():7.2f}  "
      f"min={raw_c3_before_uv.min():+8.2f}  max={raw_c3_before_uv.max():+8.2f}")
print(f"전처리 후 (unit: std. dev.): mean={raw_c3_after.mean():+8.2f}  std={raw_c3_after.std():7.2f}  "
      f"min={raw_c3_after.min():+8.2f}  max={raw_c3_after.max():+8.2f}")

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(t, raw_c3_before_uv, color="tab:gray", linewidth=0.8)
axes[0].set_title("Before preprocessing (raw, channel C3, unit: uV)")
axes[1].plot(t, raw_c3_after, color="tab:blue", linewidth=0.8)
axes[1].set_title("After bandpass 4-38Hz + standardization (channel C3, unit: std. dev.)")
axes[1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

# 시간 파형보다 주파수 성분(파워 스펙트럼)으로 보면 필터 효과가 훨씬 뚜렷하게 보입니다.
from mne.time_frequency import psd_array_welch

psd_before, freqs_before = psd_array_welch(
    raw_c3_before_uv, sfreq=sfreq_before, fmin=0, fmax=60, n_per_seg=256, verbose=False
)
psd_after, freqs_after = psd_array_welch(
    raw_c3_after, sfreq=sfreq_before, fmin=0, fmax=60, n_per_seg=256, verbose=False
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(freqs_before, psd_before, label="before", color="tab:gray")
ax.semilogy(freqs_after, psd_after, label="after", color="tab:blue")
ax.axvspan(4, 38, color="tab:green", alpha=0.1, label="kept band (4-38Hz)")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("power (log scale)")
ax.set_title("Power spectrum before vs after bandpass filtering")
ax.legend()
plt.tight_layout()
plt.show()

## 3. 이벤트 기준 윈도잉(windowing)

EEG는 계속 흐르는 하나의 긴 신호지만, 실제로 분류하고 싶은 건 "이 사람이 특정 순간에 어떤 동작을 상상했는가"입니다. 그래서 실험 중 동작을 지시한 이벤트(큐) 시점을 기준으로 일정 길이만큼 잘라낸 조각을 trial(또는 window)이라고 부릅니다.

**실험 중 피험자는 실제로 이렇게 행동합니다** (BCI Competition IV 2a 프로토콜, Tangermann et al. 2012 / moabb 문서 기준): 트라이얼 시작(t=0s)에 화면에 십자가(fixation cross)가 뜨고 짧은 경고음이 납니다. 2초 후(t=2s) 화살표 모양의 큐(좌/우/아래/위 = 왼손/오른손/발/혀)가 나타나 1.25초간 화면에 유지되고, 이 순간부터 피험자는 실제로 몸을 움직이지 않고 지시받은 동작을 상상(motor imagery)합니다. 화면 피드백은 없고, 십자가가 사라지는 t=6s까지 상상을 계속 유지합니다. 즉 큐가 뜬 시점(t=2s)부터 4초간(t=2~6s)이 실제 '상상 수행' 구간이고, 이게 annotation의 duration=4.0s와 정확히 일치합니다.

이 데이터셋은 큐(동작 상상 지시) 구간 자체가 **4.0초**이고(`raw.annotations.duration`으로 확인 가능), 여기서는 `trial_start_offset_samples=-0.5s`를 줘서 **큐 시작 0.5초 전부터 큐가 끝나는 시점(+4.0s)까지, 총 4.5초**를 잘라냅니다. 즉 "큐 이후 0.5초부터 4.5초까지"가 아니라 "큐 기준 -0.5초부터 +4.0초까지"입니다.

이게 바로 `EEG.read()`가 매번 반환해야 하는 "한 샘플"의 shape이고, **22개 전극의 데이터가 전부 하나의 (채널, 시간) 텐서로 같이 들어갑니다** (전극별로 따로 처리하는 게 아닙니다). 모델은 이 텐서 하나를 보고 클래스를 맞춥니다.

In [ ]:
display(Markdown(
    "### 큐(cue) 실험 — 실제 설정 사진 + 정확한 타이밍 (둘 다 CC BY)\n\n"
    "**A) 실제 셋업 + 화면 큐**: 사람이 실제 EEG 캡을 쓰고 앉아 있고, 정면 모니터에 어떤 동작을 하라는 지시(아이콘+화살표)가 뜹니다. "
    "출처: Triana-Guzmán, N. *et al.* \"Decoding EEG rhythms offline and online during motor imagery for standing and sitting "
    "based on a brain-computer interface.\" *Frontiers in Neuroinformatics* **16**, 961089 (2022). "
    "https://doi.org/10.3389/fninf.2022.961089 — License: [CC BY](https://creativecommons.org/licenses/by/4.0/). "
    "(주의: 이 논문은 서기/앉기 모터 이미저리 실험이라 우리 데이터셋의 왼손/오른손/발/혀 큐는 아니고, "
    "'사람이 캡을 쓰고 화면의 동작 지시를 본다'는 구성만 같아서 예시로 가져왔습니다.)\n\n"
    "**B) 정확한 타이밍**: 우리가 실제 쓰는 BCI IV 2a의 Beep→Fixation cross→Cue→Motor imagery→Break 구조와 초 단위가 그대로 일치합니다. "
    "출처: Li, X., Chen, P., Bao, Z. \"A SPA-based Manifold Learning Framework for Motor Imagery EEG Data Classification.\" "
    "Preprint (2021), Figure 1: \"BCI competition IV dataset 2a experimental paradigm.\" "
    "License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) (ResearchGate에 명시).\n\n"
    "두 이미지를 위아래로 합쳐서 하나로 보여줍니다."
))

from IPython.display import Image

display(Image(filename="cue_setup_and_timing.png"))

In [ ]:
sfreq = dataset.datasets[0].raw.info["sfreq"]
trial_start_offset_samples = int(-0.5 * sfreq)

# 큐(동작 상상 지시) 구간 자체의 길이를 실제로 확인 (추측 아님)
cue_duration_s = dataset.datasets[0].raw.annotations.duration[0]
print(f"큐 구간 길이: {cue_duration_s}s  ->  윈도우 = 큐 시작 -0.5s ~ +{cue_duration_s}s (총 {0.5 + cue_duration_s}s)")

windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=trial_start_offset_samples,
    trial_stop_offset_samples=0,
    preload=True,
)

X0, y0, crop_inds = windows_dataset[0]
print("한 샘플 X shape (n_chans, n_times):", X0.shape)
print("레이블 y (클래스 인덱스):", y0)
print("crop_inds (i_window_in_trial, i_start, i_stop):", crop_inds)

In [ ]:
display(Markdown(
    "### 이 라벨이 실제로 어떤 동작인지, 윈도우가 어떤 모양인지 확인\n\n"
    "히트맵(색으로 표현)과 그 밑에 **실제 숫자 행렬**을 같이 보여줍니다 — 히트맵의 색이 결국 이 숫자들이라는 걸 바로 확인할 수 있습니다. "
    "22채널×1125시점을 전부 보여주면 너무 많아서, 앞부분과 끝부분만 보이게 중간을 `...`로 줄였습니다 (판다스가 자동으로 이렇게 줄여줍니다). "
    "점선(t=0.5s)은 동작 상상 큐(cue)가 시작되는 시점입니다."
))

LABEL_NAMES = ["feet", "left_hand", "right_hand", "tongue"]
print(f"이 샘플의 라벨: target={y0} -> '{LABEL_NAMES[y0]}'")

cue_onset_s = 0.5  # 윈도우가 큐 시작 -0.5s부터 시작하므로, 이 그래프의 x축 기준으로 큐 시작은 0.5s 지점
ch_names = raw.ch_names  # X0의 채널 순서와 동일한 22개 전극 이름
times = np.arange(X0.shape[1]) / sfreq

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    X0, aspect="auto", cmap="RdBu_r", vmin=-4, vmax=4,
    extent=[0, X0.shape[1] / sfreq, X0.shape[0], 0],
)
ax.axvline(cue_onset_s, color="black", linestyle="--", linewidth=1, label="cue onset")
ax.set_yticks(np.arange(len(ch_names)) + 0.5)
ax.set_yticklabels(ch_names, fontsize=7)
ax.set_xlabel("time (s, 0 = window start = cue onset - 0.5s)")
ax.set_title(f"One window, all 22 channels, label='{LABEL_NAMES[y0]}'", pad=12)
ax.legend(loc="upper right")
plt.colorbar(im, ax=ax, label="standardized amplitude")
plt.tight_layout()
plt.show()

# 위 히트맵과 완전히 같은 데이터를, 실제 숫자 행렬로도 봅니다 (앞/뒤만 보이게 자동 축약)
import pandas as pd

with pd.option_context(
    "display.max_rows", 12, "display.max_columns", 10, "display.float_format", lambda v: f"{v:.2f}"
):
    df = pd.DataFrame(X0, index=ch_names, columns=[f"{t:.2f}s" for t in times])
    display(df)

In [ ]:
display(Markdown(
    "### 뇌 영역별 신호 세기 애니메이션 (scalp topomap)\n\n"
    "히트맵/행렬은 '채널 순서 x 시간'으로 나열된 숫자일 뿐, 어느 전극이 머리 위 어디에 있는지는 안 보여줍니다. "
    "여기서는 같은 데이터를 매 시점마다 두피 위 실제 전극 위치에 색으로 얹어서(topomap), 전극 이름도 점 오른쪽에 작게 같이 표시하고, "
    "슬라이더로 시간을 넘겨가며 볼 수 있게 만들었습니다. "
    "(참고: 이 스타일은 뇌파 논문에서 흔히 쓰는 표준 scalp topomap이며, 특정 논문의 그림을 그대로 가져온 것은 아닙니다 — mne로 직접 그렸습니다.)"
))

n_frames = 40
frame_indices = np.linspace(0, X0.shape[1] - 1, n_frames).astype(int)
vmax = np.abs(X0).max()

fig, (ax_topo, ax_cbar) = plt.subplots(1, 2, figsize=(5.5, 4.5), gridspec_kw={"width_ratios": [10, 1]})

def update(i):
    ax_topo.clear()
    ax_cbar.clear()
    idx = frame_indices[i]
    im, _ = mne.viz.plot_topomap(
        X0[:, idx], raw.info, axes=ax_topo, show=False,
        vlim=(-vmax, vmax), cmap="RdBu_r", sensors=True, names=ch_names,
    )
    for t in ax_topo.texts:
        x, y = t.get_position()
        t.set_position((x + 0.0025, y))  # mne.viz.plot_sensors(show_names=True)와 동일한 오프셋
        t.set_ha("left")
        t.set_fontsize(7)
    plt.colorbar(im, cax=ax_cbar, label="standardized amplitude")
    ax_topo.set_title(f"t={idx / sfreq:.2f}s  label='{LABEL_NAMES[y0]}'", fontsize=10)
    fig.tight_layout()  # 컬러바 라벨이 잘리지 않도록 매 프레임 여백 재조정

anim = animation.FuncAnimation(fig, update, frames=n_frames, interval=150)
plt.close(fig)
HTML(anim.to_jshtml())

## 4. train/valid 분할 (session 기준)

In [ ]:
splitted = windows_dataset.split("session")
print("session keys:", list(splitted.keys()))

train_key, valid_key = list(splitted.keys())[:2]
train_set, valid_set = splitted[train_key], splitted[valid_key]
print(f"train trials: {len(train_set)}, valid trials: {len(valid_set)}")

n_chans, input_window_samples = X0.shape[0], X0.shape[1]
n_classes = len(set(windows_dataset.get_metadata()["target"]))
print(f"n_chans={n_chans}, n_times={input_window_samples}, n_classes={n_classes}")

In [ ]:
display(Markdown("### 클래스 분포 확인 — 4개 클래스가 얼마나 균형있게 들어있는지 눈으로 확인합니다."))

from collections import Counter

train_counts = Counter(train_set[i][1] for i in range(len(train_set)))
valid_counts = Counter(valid_set[i][1] for i in range(len(valid_set)))

x = np.arange(len(LABEL_NAMES))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width / 2, [train_counts[i] for i in range(4)], width, label="train")
ax.bar(x + width / 2, [valid_counts[i] for i in range(4)], width, label="valid")
ax.set_xticks(x)
ax.set_xticklabels(LABEL_NAMES)
ax.set_ylabel("number of trials")
ax.set_title("Trials per class (4-class motor imagery)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. `EEGNet` 디코더 구조 및 input/output shape

현재 braindecode에서 이 모델 클래스 이름은 `EEGNet` (예전 이름은 `EEGNetv4`), 생성자 파라미터는 `n_chans`, `n_outputs`, `n_times`.

In [ ]:
model = EEGNet(n_chans=n_chans, n_outputs=n_classes, n_times=input_window_samples)
print(model)

In [ ]:
device = next(model.parameters()).device
print("model device:", device)

model.eval()
with torch.no_grad():
    X_batch = torch.tensor(X0).unsqueeze(0).float().to(device)
    print("model input shape (batch, n_chans, n_times):", tuple(X_batch.shape))
    out = model(X_batch)
    print("model output shape (batch, n_classes):", tuple(out.shape))
    print("raw logits:", out)
    print("note: 현재 버전은 마지막에 LogSoftmax가 없고 raw logit을 그대로 반환함 -> loss는 CrossEntropyLoss를 써야 함")

In [ ]:
display(Markdown("### 모델 출력을 그림으로 보기 — raw logit과 softmax 확률이 클래스별로 어떻게 나오는지 확인합니다."))

probs = torch.softmax(out, dim=1)[0].detach().cpu().numpy()
logits_np = out[0].detach().cpu().numpy()
pred_idx = int(logits_np.argmax())
colors = ["tab:orange" if i == pred_idx else "tab:blue" for i in range(4)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(LABEL_NAMES, logits_np, color=colors)
axes[0].set_title("raw logit")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(LABEL_NAMES, probs, color=colors)
axes[1].set_title("softmax probability")
axes[1].tick_params(axis="x", rotation=20)

fig.suptitle(
    f"Predicted: '{LABEL_NAMES[pred_idx]}' (true label: '{LABEL_NAMES[y0]}') "
    "- untrained model, close to random"
)
plt.tight_layout()
plt.show()

## 6. 간단한 학습 sanity check (`CrossEntropyLoss` 사용)

이전 스모크 테스트에서는 `NLLLoss`를 잘못 써서 음수 loss가 나왔습니다 (raw logit에 NLLLoss를 쓰면 잘못된 값이 나옴). 여기서는 올바르게 `CrossEntropyLoss`를 사용합니다.

In [ ]:
model.train()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

n_batch = min(16, len(train_set))
xs = np.stack([train_set[i][0] for i in range(n_batch)])
ys = np.array([train_set[i][1] for i in range(n_batch)])
X_batch = torch.from_numpy(xs).float().to(device)
y_batch = torch.from_numpy(ys).long().to(device)

n_steps = 40
history = {"loss": [], "acc": []}
for step in range(n_steps):
    opt.zero_grad()
    logits = model(X_batch)
    loss = loss_fn(logits, y_batch)
    loss.backward()
    opt.step()
    acc = (logits.argmax(dim=1) == y_batch).float().mean().item()
    history["loss"].append(loss.item())
    history["acc"].append(acc)

print(f"마지막 step: loss={history['loss'][-1]:.4f}, acc={history['acc'][-1]:.2f}")
print("주의: 딱 16개 trial짜리 배치 하나만 반복 학습시킨 것이라, '제대로 된 학습'이 아니라 오버피팅이 되는지 보는 sanity check입니다.")

In [ ]:
display(Markdown("### 학습 곡선 그려보기 — loss가 떨어지고 정확도가 올라가는 걸 곡선으로 보면 훨씬 직관적입니다. (16개 고정 배치를 반복 학습한 오버피팅 sanity check라는 점은 유의하세요.)"))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history["loss"])
axes[0].set_title("loss (lower is better)")
axes[0].set_xlabel("step")

axes[1].plot(history["acc"])
axes[1].set_title("accuracy (on this fixed 16-sample batch)")
axes[1].set_xlabel("step")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()